This notebook is an in-depth analysis of the mobsc dataset. The main observation is that the original data had been normalized using scanpy, and using a negative binomial version of the original counts data does better than a truncated zero-inflated Gaussian on the normalized data. 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
import scanpy as sc


def compare_pca(
    real,
    simulated,
    *,
    color=None,
    n_pcs=50,
    random_state=42,
):
    """Fit PCA on real cells and compare real and simulated projections."""
    real_projection = real.copy()
    sim_projection = simulated[:, real.var_names].copy()

    for adata in (real_projection, sim_projection):
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

    n_components = min(
        n_pcs, real_projection.n_obs - 1, real_projection.n_vars - 1
    )
    real_matrix = (
        real_projection.X.toarray()
        if hasattr(real_projection.X, "toarray")
        else np.asarray(real_projection.X)
    )
    sim_matrix = (
        sim_projection.X.toarray()
        if hasattr(sim_projection.X, "toarray")
        else np.asarray(sim_projection.X)
    )

    # Fit on real cells and transform simulated cells with the same PCA model.
    pca = PCA(n_components=n_components, random_state=random_state)
    real_projection.obsm["X_pca"] = pca.fit_transform(real_matrix)
    sim_projection.obsm["X_pca"] = pca.transform(sim_matrix)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    sc.pl.pca(
        real_projection, color=color, title="Real", ax=axes[0], show=False
    )
    sc.pl.pca(
        sim_projection, color=color, title="Simulated", ax=axes[1], show=False
    )

    coordinates = np.vstack(
        (
            real_projection.obsm["X_pca"][:, :2],
            sim_projection.obsm["X_pca"][:, :2],
        )
    )
    x_padding = max(0.05 * np.ptp(coordinates[:, 0]), 0.1)
    y_padding = max(0.05 * np.ptp(coordinates[:, 1]), 0.1)
    for ax in axes:
        ax.set_xlim(
            coordinates[:, 0].min() - x_padding,
            coordinates[:, 0].max() + x_padding,
        )
        ax.set_ylim(
            coordinates[:, 1].min() - y_padding,
            coordinates[:, 1].max() + y_padding,
        )

    return fig, axes

In [ ]:
from scdesigner.datasets import mobsc
MOBSC_sce = mobsc()

MOBSC_sce.obs

In [ ]:
from scdesigner.simulators import NegBinCopula

sim = NegBinCopula(
    mean_formula = "~ cell_type",
    dispersion_formula = "~ cell_type"
)
sim.fit(MOBSC_sce)

In [ ]:
sampled = sim.sample()
compare_pca(MOBSC_sce, sampled, color="cell_type")

In [ ]:
compare_pca(MOBSC_sce, sampled, color="sampleInfo")

In [ ]:
compare_pca(MOBSC_sce, sampled, color="sizeFactor")

In [ ]:
sim = NegBinCopula(
    mean_formula = "~ cell_type + sizeFactor",
    dispersion_formula = "~ cell_type + sizeFactor"
)
sim.fit(MOBSC_sce)

In [ ]:
sampled = sim.sample()
compare_pca(MOBSC_sce, sampled, color="sizeFactor")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# NOTE: adata.uns["X_name"] == "counts" for MOBSC_sce, but .X holds non-integer
# floats (e.g. 1.93, 2.54) that don't match layers["logcounts"] either, so this
# is not raw count data despite the label -- plotted as provided.
_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]


def _dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.asarray(X)


def _select_genes(adata, X, genes, n_genes, rank_by, ascending):
    if genes is not None:
        idx = np.array([adata.var_names.get_loc(g) for g in genes])
        return idx, np.asarray(genes)

    if callable(rank_by):
        scores = rank_by(X)
    elif rank_by == "variance":
        scores = X.var(axis=0)
    elif rank_by == "mean":
        scores = X.mean(axis=0)
    else:
        raise ValueError(f"Unknown rank_by: {rank_by!r}")

    order = np.argsort(scores)
    if not ascending:
        order = order[::-1]
    idx = order[:n_genes]
    return idx, adata.var_names.to_numpy()[idx]


def plot_gene_scatter(
    adata,
    genes=None,
    *,
    n_genes=10,
    rank_by="variance",
    ascending=False,
    color_by="cell_type",
    x_obs="sizeFactor",
    ncols=5,
):
    """Scatter `x_obs` vs. expression for a set of genes, colored by a categorical obs column.

    Works on any AnnData with the same obs schema (real or simulated).

    genes : explicit gene names to plot; overrides n_genes/rank_by/ascending.
    n_genes, rank_by, ascending : select genes by ranking `rank_by` ("variance",
        "mean", or a callable taking the dense expression matrix and returning a
        per-gene score); ascending=True picks the lowest-scoring genes instead
        (e.g. rank_by="mean", ascending=True -> least abundant genes).
    color_by : categorical column of `adata.obs` used for point color.
    x_obs : column of `adata.obs` used for the x-axis.
    """
    for col in (x_obs, color_by):
        if col not in adata.obs:
            raise ValueError(f"{col!r} is not present in adata.obs.")

    X = _dense(adata.X)
    gene_idx, gene_names = _select_genes(adata, X, genes, n_genes, rank_by, ascending)
    if len(gene_idx) == 0:
        raise ValueError("No genes selected.")

    x_values = adata.obs[x_obs].to_numpy()
    color_values = adata.obs[color_by].astype("category")
    categories = list(color_values.cat.categories)
    color_by_cat = {cat: _PALETTE[i % len(_PALETTE)] for i, cat in enumerate(categories)}

    ncols = min(ncols, len(gene_idx))
    nrows = int(np.ceil(len(gene_idx) / ncols))
    figsize = (4 * ncols, 4 * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, constrained_layout=True, squeeze=False)
    axes_flat = axes.flat

    for ax, gi, gene_name in zip(axes_flat, gene_idx, gene_names):
        expr = X[:, gi]
        for cat in categories:
            mask = (color_values == cat).to_numpy()
            ax.scatter(
                x_values[mask], expr[mask],
                s=8, alpha=0.6, color=color_by_cat[cat],
                edgecolors="none", label=cat,
            )
        ax.set_title(gene_name, fontsize=10)
        ax.set_xlabel(x_obs)
        ax.set_ylabel("expression")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # Hide any unused axes when n_genes isn't a multiple of ncols.
    for ax in axes_flat[len(gene_idx):]:
        ax.set_visible(False)

    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(
        handles, labels, title=color_by, loc="upper center",
        bbox_to_anchor=(0.5, 1 + 0.5 / figsize[1]), ncol=len(categories), frameon=False,
    )
    fig.suptitle(f"{x_obs} vs. expression", y=1 + 0.9 / figsize[1])
    return fig, axes

In [ ]:
# real data
plot_gene_scatter(MOBSC_sce, n_genes=10, rank_by="variance", color_by="cell_type")
plt.show()

In [ ]:
# simulated data
plot_gene_scatter(sampled, n_genes=10, rank_by="variance", color_by="cell_type")
plt.show()

In [ ]:
# New gene subset: 10 least abundant genes instead of the most variable ones.
plot_gene_scatter(MOBSC_sce, n_genes=10, rank_by="mean", ascending=True, color_by="cell_type")
plt.show()

In [ ]:
plot_gene_scatter(MOBSC_sce, n_genes=10, rank_by="variance", color_by="sampleInfo")
plt.show()

In [ ]:
# Our data were transformed using log1p(count / total_counts * 1e4).
# This inverts the transformation back to count space.

E = np.expm1(_dense(MOBSC_sce.X))
total_counts = 1e4 / np.where(E > 0, E, np.inf).min(axis=1)
counts = np.round(E * total_counts[:, None] / 1e4)
assert np.allclose(np.log1p(counts / total_counts[:, None] * 1e4), _dense(MOBSC_sce.X))
assert np.allclose(total_counts, np.round(total_counts))

MOBSC_counts = MOBSC_sce.copy()
MOBSC_counts.X = counts
MOBSC_counts.obs["total_counts"] = total_counts
MOBSC_counts.obs["log_total"] = np.log(total_counts)

In [ ]:
sim_counts = NegBinCopula(
    mean_formula="~ cell_type + log_total",
    dispersion_formula="~ cell_type",
)
sim_counts.fit(MOBSC_counts)
sampled_counts = sim_counts.sample()

In [ ]:
def lognorm(adata):
    """Apply the same scanpy log1p(count / total_counts * 1e4) transformation"""
    out = adata.copy()
    total = adata.obs["total_counts"].to_numpy()[:, None]
    out.X = np.log1p(_dense(adata.X) / total * 1e4)
    return out


# plot the simulated data after transformation
plot_gene_scatter(lognorm(sampled_counts), n_genes=10, rank_by="variance", color_by="cell_type")
plt.show()

compare_pca(MOBSC_counts, sampled_counts, color="cell_type")
plt.show()

In [ ]:
compare_pca(MOBSC_counts, sampled_counts, color="sizeFactor")

In [ ]:
# reclustering
gc = MOBSC_sce[MOBSC_sce.obs["cell_type"] == "GC"].copy()
sc.pp.pca(gc, n_comps=20)
sc.pp.neighbors(gc, n_neighbors=15, n_pcs=20)
sc.tl.leiden(gc, resolution=0.3, key_added="gc_cluster", flavor="igraph", n_iterations=2, random_state=0)

cell_type2 = MOBSC_counts.obs["cell_type"].astype(str)
cell_type2.loc[gc.obs_names] = "GC_" + gc.obs["gc_cluster"].astype(str)
MOBSC_counts.obs["cell_type2"] = cell_type2.astype("category")

In [ ]:
sim_counts2 = NegBinCopula(
    mean_formula="~ cell_type2 + log_total",
    dispersion_formula="~ cell_type2",
)
sim_counts2.fit(MOBSC_counts)
sampled_counts2 = sim_counts2.sample()

compare_pca(MOBSC_counts, sampled_counts2, color="cell_type2")

In [ ]:
plot_gene_scatter(lognorm(sampled_counts2), n_genes=10, rank_by="variance", color_by="cell_type")

In [ ]:
from pathlib import Path

MOBSC_out = MOBSC_counts.copy()
MOBSC_out.obs["cell_type"] = MOBSC_out.obs.pop("cell_type2")
MOBSC_out.uns["X_name"] = "counts"

out_path = Path("data/mobsc_counts.h5ad")
out_path.parent.mkdir(exist_ok=True)
MOBSC_out.write_h5ad(out_path)
MOBSC_out.obs["cell_type"].value_counts()